# 2026-10-18_3 scTenifoldKnk Figure 5：Target-level virtual KO dot plot

基于 target 配对参数表（stage 拆分行）绘制 Figure 5：三对 TF-target 配对的 KO 响应 dot plot，不修改任何原始分析结果。

- 数据源说明：任务指定的 `2026-10-16_stage_split_target_pairs_params (2).csv` 在服务器上不存在，使用其内容一致的正式文件 `2026-10-16_stage_split_target_pairs_params.csv`（三对配对 12 行 stage 数据与 (2) 版本 FDR 值逐一相符：3.94e-06 / 4.36e-10 / 5.14e-06）
- Y 轴固定三组（subclass 分类已从 DEG 文件核实）：ONECUT1 (MG) -> ABCB11 (DAL)、ONECUT1 (MG) -> TTC39B (DM)、MAFK (MG) -> TFRC (AAM)
- X 轴固定顺序：Central Young、Central Old、Peripheral Young、Peripheral Old（按 Central/Peripheral 分面，分面内 Young/Old）
- 圆点大小 = -log10(p.adj)，显示上限 10（数据最大 9.36）
- 颜色区分预测方向 Down/Up（direction 字段，KO 预测响应方向，非表达变化）；显著（p.adj<0.05）实心圆并直接标注 FDR；不显著空心圆标注 ns；未运行标注 NA（当前 12 行均有数值，NA 分支已实现备用）
- Y 轴标签中箭头使用 ASCII "->"（PDF Helvetica 字体不含 Unicode 箭头，避免字形丢失）
- 输出：`Plot/2026-10-18_Fig5_target_level_virtual_KO_dotplot.pdf`（仅 PDF，白底简洁论文风格）

In [ ]:
# ---- 1. 配置：路径 / 配对 / 颜色（集中修改处） --------------------------------
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
})
set.seed(1)

KNK_DIR  <- "/project/work/2026-10-13_scTenifoldKnk/Data/"
PLOT_DIR <- "/project/work/2026-10-13_scTenifoldKnk/Plot/"
DATE     <- "2026-10-18"
PADJ_CUTOFF <- 0.05
SIZE_CAP    <- 10    # -log10(p.adj) 显示上限（数据最大 9.36）

# 数据源（任务指定的 "(2).csv" 服务器上不存在，使用内容一致的正式文件）
PARAMS_CSV <- file.path(KNK_DIR, "2026-10-16_stage_split_target_pairs_params.csv")

# 颜色（与 2026-10-18_2 显著基因计数图一致，color.ipynb palette3 中间调）
cols.direction <- c(down = "#8491B4", up = "#F39B7F")

# Y 轴三组配对（subclass 已从 DEG 文件核实：ABCB11=DAL, TTC39B=DM, TFRC=AAM）
pair_spec <- data.table(
  TF          = c("ONECUT1", "ONECUT1", "MAFK"),
  target_gene = c("ABCB11", "TTC39B", "TFRC"),
  celltype    = "MG",
  subclass    = c("DAL", "DM", "AAM")
)

## 读取参数表，提取 12 行（3 配对 x young/old x central/peripheral，固定顺序）
stopifnot(file.exists(PARAMS_CSV))
params <- fread(PARAMS_CSV)
grid <- rbindlist(lapply(seq_len(nrow(pair_spec)), function(i) {
  cbind(pair_spec[i, .(TF, target_gene, celltype, subclass)],
        CJ(region = c("central", "peripheral"), stage = c("young", "old")))
}))
d <- merge(grid, params[, .(TF, target_gene, celltype, region, stage, p.adj, direction)],
           by = c("TF", "target_gene", "celltype", "region", "stage"), all.x = TRUE)

## 状态判定：sig（p.adj<0.05）/ ns / NA（未运行）；大小 = -log10(p.adj) 封顶
d[, padj_num := as.numeric(p.adj)]
d[, status := fcase(is.na(padj_num), "NA", padj_num < PADJ_CUTOFF, "sig", default = "ns")]
d[, neglog := ifelse(is.na(padj_num), NA_real_, pmin(-log10(padj_num), SIZE_CAP))]

## 坐标标签
d[, pair_lab := sprintf("%s (%s) -> %s (%s)", TF, celltype, target_gene, subclass)]
d[, pair_lab := factor(pair_lab, levels = rev(sprintf("%s (%s) -> %s (%s)",
    pair_spec$TF, pair_spec$celltype, pair_spec$target_gene, pair_spec$subclass)))]  # 自下而上
d[, Region_lab := factor(c(central = "Central", peripheral = "Peripheral")[region],
                         levels = c("Central", "Peripheral"))]
d[, Stage_lab := factor(c(young = "Young", old = "Old")[stage], levels = c("Young", "Old"))]
d <- d[order(pair_lab, Region_lab, Stage_lab)]

cat("绘图数据（12 行，固定 X 顺序）:\n")
print(d[, .(pair_lab, Region_lab, Stage_lab, direction, p.adj = signif(padj_num, 3),
            status, neglog = round(neglog, 2))])
cat("\n状态分布:", paste(d[, .N, by = status][, sprintf("%s=%d", status, N)], collapse = " "), "\n")

In [ ]:
# ---- 2. Figure 5: Target-level virtual KO dot plot ----------------------------
p <- ggplot(d, aes(Stage_lab, pair_lab)) +
  # 显著：实心圆（大小 = -log10(p.adj) 封顶，颜色 = 预测方向）
  geom_point(data = d[status == "sig"],
             aes(size = neglog, color = direction), shape = 16) +
  # 不显著：空心圆
  geom_point(data = d[status == "ns"],
             aes(size = neglog, color = direction), shape = 1, stroke = 0.9) +
  # 显著点直接标注 FDR（置于点下方）
  geom_text(data = d[status == "sig"],
            aes(y = as.numeric(pair_lab) - 0.34,
                label = sprintf("FDR = %.2e", padj_num)),
            size = 2.5, color = "grey15", lineheight = 0.9) +
  # 不显著标注 ns
  geom_text(data = d[status == "ns"],
            aes(y = as.numeric(pair_lab) - 0.26, label = "ns"),
            size = 2.3, color = "grey45") +
  # 未运行标注 NA（当前数据无此类行，分支备用）
  geom_text(data = d[status == "NA"],
            aes(label = "NA"), size = 2.8, color = "grey45") +
  facet_grid(. ~ Region_lab) +  # 两分面 stage 水平相同，用固定 x 刻度（free_x 会使各面板按子集层训练顺序生成刻度，导致顺序不稳定）
  scale_x_discrete(drop = FALSE) +
  scale_color_manual(values = cols.direction,
                     labels = c(down = "Predicted Down", up = "Predicted Up"),
                     name = NULL, drop = FALSE) +
  scale_size_continuous(limits = c(0, SIZE_CAP), breaks = c(0, 5, 10),
                        range = c(1.6, 7.5),
                        name = expression("-log10(p.adj)" ~ "(capped at 10)")) +
  scale_y_discrete(expand = expansion(add = 0.55)) +
  labs(title = "Target-level responses to virtual knockout",
       x = NULL, y = NULL) +
  theme_bw(base_size = 11) +
  theme(legend.position = "right",
        panel.grid.minor = element_blank(),
        strip.text = element_text(size = 10, face = "bold"),
        axis.text.y = element_text(size = 9.5, color = "black"))
print(p)
ggsave(file.path(PLOT_DIR, paste0(DATE, "_Fig5_target_level_virtual_KO_dotplot.pdf")),
       p, width = 8, height = 3.8)
cat("已保存:", DATE, "_Fig5_target_level_virtual_KO_dotplot.pdf\n")